In [1]:
import os.path as osp
import torch
from torch.nn import Linear

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GraphConv, dense_mincut_pool
from torch_geometric import utils
from torch_geometric.nn import Sequential
from torch_geometric.nn.conv.gcn_conv import gcn_norm

from sklearn.metrics import normalized_mutual_info_score as NMI

torch.manual_seed(0) # for reproducibility

In [2]:
# Load data
dataset = 'Cora'
#path = osp.join(osp.dirname(osp.realpath(__file__)), '.', 'data', dataset)
path = osp.join(osp.abspath(''), '..', 'data', 'cora_dataset')
dataset = Planetoid(path, dataset, transform=T.NormalizeFeatures())
data = dataset[0]

# Normalized adjacency matrix
data.edge_index, data.edge_weight = gcn_norm(  
                data.edge_index, data.edge_weight, data.num_nodes,
                add_self_loops=False, dtype=data.x.dtype)

In [5]:
dataset, len(dataset)

(Cora(), 1)

In [6]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_weight=[10556])

In [4]:
class Net(torch.nn.Module):
    def __init__(self, 
                 mp_units,
                 mp_act,
                 in_channels, 
                 n_clusters, 
                 mlp_units=[],
                 mlp_act="Identity"):
        super().__init__()
        
        mp_act = getattr(torch.nn, mp_act)(inplace=True)
        mlp_act = getattr(torch.nn, mlp_act)(inplace=True)
        
        # Message passing layers
        mp = [
            (GraphConv(in_channels, mp_units[0]), 'x, edge_index, edge_weight -> x'),
            mp_act
        ]
        for i in range(len(mp_units)-1):
            mp.append((GraphConv(mp_units[i], mp_units[i+1]), 'x, edge_index, edge_weight -> x'))
            mp.append(mp_act)
        self.mp = Sequential('x, edge_index, edge_weight', mp)
        out_chan = mp_units[-1]
        
        # MLP layers
        self.mlp = torch.nn.Sequential()
        for units in mlp_units:
            self.mlp.append(Linear(out_chan, units))
            out_chan = units
            self.mlp.append(mlp_act)
        self.mlp.append(Linear(out_chan, n_clusters))
        

    def forward(self, x, edge_index, edge_weight):
        
        # Propagate node feats
        x = self.mp(x, edge_index, edge_weight) 
        
        # Cluster assignments (logits)
        s = self.mlp(x) 
        
        # Obtain MinCutPool losses
        adj = utils.to_dense_adj(edge_index, edge_attr=edge_weight)
        _, _, mc_loss, o_loss = dense_mincut_pool(x, adj, s)
        
        return torch.softmax(s, dim=-1), mc_loss, o_loss


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
model = Net([16], "ELU", dataset.num_features, dataset.num_classes).to(device)
print(model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

Net(
  (mp): Sequential(
    (0) - GraphConv(1433, 16): x, edge_index, edge_weight -> x
    (1) - ELU(alpha=1.0, inplace=True): x -> x
  )
  (mlp): Sequential(
    (0): Linear(in_features=16, out_features=7, bias=True)
  )
)


In [6]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_weight=[10556])

In [7]:
data.x[122].shape, data.y[122]

(torch.Size([1433]), tensor(6, device='cuda:0'))

In [8]:
def train():
    model.train()
    optimizer.zero_grad()
    _, mc_loss, o_loss = model(data.x, data.edge_index, data.edge_weight)
    loss = mc_loss + o_loss
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def test():
    model.eval()
    clust, _, _ = model(data.x, data.edge_index, data.edge_weight)
    return NMI(clust.max(1)[1].cpu(), data.y.cpu()), clust

In [9]:
patience = 50
best_nmi = 0
for epoch in range(1, 10000):
    train_loss = train()
    nmi, clust = test()
    print(f'Epoch: {epoch:03d}, Loss: {train_loss:.4f}, NMI: {nmi:.3f}')
    if nmi > best_nmi:
        best_nmi = nmi
        patience = 50
    else:
        patience -= 1     
    if patience == 0:
        break

Epoch: 001, Loss: 0.1154, NMI: 0.000
Epoch: 002, Loss: 0.1154, NMI: 0.001
Epoch: 003, Loss: 0.1154, NMI: 0.012
Epoch: 004, Loss: 0.1154, NMI: 0.069
Epoch: 005, Loss: 0.1154, NMI: 0.040
Epoch: 006, Loss: 0.1154, NMI: 0.043
Epoch: 007, Loss: 0.1154, NMI: 0.080
Epoch: 008, Loss: 0.1154, NMI: 0.120
Epoch: 009, Loss: 0.1153, NMI: 0.134
Epoch: 010, Loss: 0.1153, NMI: 0.145
Epoch: 011, Loss: 0.1153, NMI: 0.149
Epoch: 012, Loss: 0.1152, NMI: 0.147
Epoch: 013, Loss: 0.1150, NMI: 0.142
Epoch: 014, Loss: 0.1148, NMI: 0.143
Epoch: 015, Loss: 0.1145, NMI: 0.139
Epoch: 016, Loss: 0.1140, NMI: 0.138
Epoch: 017, Loss: 0.1133, NMI: 0.138
Epoch: 018, Loss: 0.1124, NMI: 0.139
Epoch: 019, Loss: 0.1112, NMI: 0.142
Epoch: 020, Loss: 0.1097, NMI: 0.148
Epoch: 021, Loss: 0.1077, NMI: 0.155
Epoch: 022, Loss: 0.1051, NMI: 0.161
Epoch: 023, Loss: 0.1020, NMI: 0.169
Epoch: 024, Loss: 0.0983, NMI: 0.174
Epoch: 025, Loss: 0.0940, NMI: 0.177
Epoch: 026, Loss: 0.0891, NMI: 0.186
Epoch: 027, Loss: 0.0835, NMI: 0.195
E

In [10]:
clust.shape

torch.Size([2708, 7])

In [41]:
clust.max(1)[1]

tensor([1, 0, 0,  ..., 6, 1, 1], device='cuda:0')

In [38]:
data.y

tensor([3, 4, 4,  ..., 3, 3, 3], device='cuda:0')